In [1]:
%pwd

'/home/narendra/Project/Medical-ChatBot/research'

In [2]:
import os

os.chdir("../")

In [3]:
%pwd

'/home/narendra/Project/Medical-ChatBot'

In [4]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/home/narendra/Project/Medical-ChatBot/vevn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Extract text from PDF files
def load_pdf_files(data):
    loader = DirectoryLoader(data, glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()

    return documents

In [6]:
extracted_docs = load_pdf_files("data")
print(f"Number of documents extracted: {len(extracted_docs)}")

Number of documents extracted: 637


In [12]:
# fileter Documents

from typing import List
from langchain_core.documents import Document


def filter_to_minimal_docs(documents: List[Document]) -> List[Document]:
    minimal_docs = []
    for doc in documents:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(page_content=doc.page_content, metadata={"source": src})
        )
    return minimal_docs

In [13]:
minimal_docs = filter_to_minimal_docs(extracted_docs)
print(f"Number of minimal documents: {len(minimal_docs)}")

Number of minimal documents: 637


In [14]:
# split the documents into chunks


def text_splits(minimal_docs):

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    split_docs = text_splitter.split_documents(minimal_docs)
    return split_docs

In [15]:
text_chunks = text_splits(minimal_docs)
print(f"Number of text chunks: {len(text_chunks)}")

Number of text chunks: 3426


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings


def download_embeddings():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    return embeddings

In [21]:
embedding = download_embeddings()

/tmp/ipykernel_10900/2918498284.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [22]:
embed_vector = embedding.embed_query("What is the capital of France?")
print(embed_vector)
print(f"Length of embedding vector: {len(embed_vector)}")

[0.08204812556505203, 0.03605552762746811, -0.003892907639965415, -0.004881060216575861, 0.025651149451732635, -0.0571434423327446, 0.012191560119390488, 0.004678943194448948, 0.03494987636804581, -0.02242191694676876, -0.008005215786397457, -0.10935354232788086, 0.022724760696291924, -0.02932087890803814, -0.043522078543901443, -0.1202412098646164, -0.0008486167062073946, -0.018150128424167633, 0.05612951144576073, 0.0030852677300572395, 0.0023363837972283363, -0.016839269548654556, 0.06362468749284744, -0.023660218343138695, 0.031493522226810455, -0.034797899425029755, -0.020548859611153603, -0.002790990052744746, -0.011038009077310562, -0.03612672537565231, 0.0541410855948925, -0.03661715239286423, -0.025008641183376312, -0.038170382380485535, -0.049603648483753204, -0.015148092992603779, 0.021315038204193115, -0.012740406207740307, 0.07670096307992935, 0.044355735182762146, -0.010834868997335434, -0.029760031029582024, -0.016970470547676086, -0.024691862985491753, 0.008087058551609

In [ ]:
from dotenv import load_dotenv

load_dotenv()

os.environ["PINECONE_API_KEY"] = os.getenv("PINECONE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [3]:
from pinecone import Pinecone
import os

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

pc

In [4]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot-index"

# Check if index exists
existing_indexes = [i.name for i in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        serverless_spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

/home/narendra/Project/Medical-ChatBot/vevn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks, embedding=embedding, index_name=index_name
)

In [27]:
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name, embedding=embedding
)

In [28]:
docsearch

In [29]:
example = Document(
    metadata={"source": "example.pdf"},
    page_content="What is the capital of France?",
)

In [30]:
docsearch.add_documents([example])

['6fe4118b-eccb-4680-b61a-52bae3ba06d4']

In [31]:
retriver = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [32]:
retrive_docs = retriver.invoke("What is the capital of France?")

In [36]:
print(retrive_docs)
print(retrive_docs[0].metadata["source"])
print(len(retrive_docs))

[Document(id='50a07dbc-9a7a-4be3-bfef-0bd2b54566b5', metadata={'source': 'example.pdf'}, page_content='What is the capital of France?'), Document(id='7e3f2c48-46c1-4e72-8422-d22ffe867ac1', metadata={'source': 'data/Medical_book.pdf'}, page_content='• tetracyclines (used to treat infections). Using these medi-\ncines with isotretinoin increases the chance of swelling\nof the brain. Make sure the physician knows if tetracy-\ncline is being used to treat acne or another infection.\nNancy Ross-Flanigan\nKEY TERMS\nAcne—A skin condition in which raised bumps,\npimples, and cysts form on the face, neck, shoul-\nders and upper back.\nBacteria—Tiny, one-celled forms of life that cause\nmany diseases and infections.\nBowel—The intestine; a tube-like structure that\nextends from the stomach to the anus. Some diges-\ntive processes are carried out in the bowel before\nfood passes out of the body as waste.\nCyst—An abnormal sac or enclosed cavity in the\nbody, filled with liquid or partially solid

In [34]:
from langchain_groq import ChatGroq

In [35]:
model = ChatGroq(model="openai/gpt-oss-120b", temperature=0.7)

In [40]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
system_prompt = """You are a medical assistant chatbot designed to provide accurate, safe, and helpful health-related information.

Your responses MUST follow these rules:

1. Grounding in Retrieved Context:
- Only use the provided retrieved documents (context) to answer the user’s question.
- If the answer is not clearly supported by the retrieved context, say:
  "I’m not sure based on the available information."
- Do NOT fabricate or guess medical facts.

2. Safety and Medical Disclaimer:
- You are NOT a doctor.
- Always include a brief safety note when appropriate:
  "This information is for educational purposes only and not a substitute for professional medical advice."
- For serious, urgent, or unclear symptoms, advise seeking medical help.

3. No Diagnosis or Prescriptions:
- Do NOT provide definitive diagnoses.
- Do NOT prescribe medications or exact dosages.
- You may explain general treatment approaches mentioned in the context.

4. Clarity and Simplicity:
- Use simple, easy-to-understand language.
- Avoid unnecessary medical jargon, or explain it if used.

5. Structured Responses:
- When appropriate, structure answers with:
  - Brief explanation
  - Key points or symptoms
  - When to seek medical help

6. Handling Uncertainty:
- If multiple possibilities exist, present them without concluding a diagnosis.
- Use phrases like:
  "This could be related to..."
  "Some possible causes include..."

7. Respect Boundaries:
- If asked harmful, unsafe, or non-medical questions, politely refuse or redirect.

8. Context Priority:
- If the retrieved documents conflict with general knowledge, prioritize the retrieved documents.

9. Tone:
- Be calm, respectful, and supportive.
- Do NOT be alarmist or overly casual.

---

Context:
{context}
"""

In [66]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [67]:
Question_answer_chain = create_stuff_documents_chain(model, prompt)

rag_chain = create_retrieval_chain(retriver, Question_answer_chain)

In [69]:
rag_chain.invoke({"input": "What is Acne?"})

{'input': 'What is Acne?',
 'context': [Document(id='90c2b071-851b-4f4d-be5d-bc5141c4693c', metadata={'source': 'data/Medical_book.pdf'}, page_content='The goal of treating moderate acne is to decrease\ninflammation and prevent new comedone formation. One\neffective treatment is topical tretinoin along with a topical\nGALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25'),
  Document(id='f8077d29-6102-422b-a1a9-9a57e4caeea7', metadata={'source': 'data/Medical_book.pdf'}, page_content='The goal of treating moderate acne is to decrease\ninflammation and prevent new comedone formation. One\neffective treatment is topical tretinoin along with a topical\nGALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman

In [ ]:
"""================ PIPELINE STRUCTURE ================

User Input
   ↓
[Retriever.invoke]
   ↓
[List of Documents]
   ↓
[format_docs]
   ↓
[String Context]
   ↓
[ChatPromptTemplate]
   ↓
[LLM]
   ↓
[StrOutputParser]
   ↓
Final Answer



code part -->

from langchain_core.runnables import RunnablePassthrough
from langchain.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

document_chain = (
    {
        "context": lambda x: format_docs(x["context"]),
        "question": lambda x: x["input"]
    }
    | prompt
    | llm
    | StrOutputParser()
)

chain = (
    {
        "context": lambda x: retriever.invoke(x["input"]),
        "input": lambda x: x["input"]
    }
    | document_chain
)
"""

TypeError: unsupported operand type(s) for |: 'dict' and 'function'